In [1]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable


In [3]:
# make sure we are in the right folder to begin working
import os

target_folder = "CS6423_knowledge_distillation_project" 
path = os.path.join(os.getcwd(), target_folder)

if os.path.exists(path):
    os.chdir(path)

# should match the folder you cloned into
print(f"Current working directory: {os.getcwd()}")

Current working directory: /home/epoc2/CS6423_knowledge_distillation_project


### Loading the pretrained model
Function that loads weights and returns fully trained model - our teacher - as pre_model

In [3]:
import torch
import torch.nn as nn
from torchvision import models
import argparse
from collections import OrderedDict
from torchsummary import summary

# define function to initialize and return our pretrained model - the resnet50
def load_pre_model(checkpoint_path, device, num_classes):
    pre_model = models.resnet50(weights=None)
    pre_model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(2048, num_classes)
    )

    torch.serialization.add_safe_globals([argparse.Namespace])
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    state_dict = checkpoint['model']
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k.replace('_orig_mod.', '') 
        new_state_dict[name] = v

    pre_model.load_state_dict(new_state_dict, strict=True)
    
    pre_model.eval()
    for p in pre_model.parameters():
        p.requires_grad = False

    # send to gpu and output summary
    pre_model.to(device)
    # summary(pre_model, input_size=(3, 224, 224))
    return pre_model


Function for saving our supernet model as we train it

In [4]:
# define a function to save our supernet model
def save_supernet(model, path="supernet_checkpoint.pth"):
    # we'll save the state dict along with the search space for future reference
    torch.save({
        'state_dict': model.state_dict(),
        'width_mult_list': [0.65, 0.8, 1.0, 1.2],
        'search_space': {
            'res': [128, 160, 190, 224],
            'depth': [2, 3, 4],
            'exp': [3, 4, 6]
        }
    }, path)
    print(f"supernet saved to {path}")

### Preparing Data Loaders
We need to provide data loaders to our training function.  We need:
* training loader - used to distill knowledge into supernet
* search/validation loader - used by the genetic algorithm later to calculate accuracy without any bias
* test loader - final test set to verify performance of student model

Here we'll initialize them.

In [5]:
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

# fiachra's transform functions from modules data prepepr
def get_train_transform(self):
    return transforms.Compose(
        [
            transforms.Resize(256),
            transforms.RandomRotation(15),
            transforms.RandomHorizontalFlip(),
            transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
        ]
    )

def get_val_transform(self):
    return transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
        ]
    )


def get_data_loaders(data_dir, batch_size=32):
    # transforms - ought to match those applied to our dta during resnet fine-tuning
    train_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])

    val_test_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])

    # load actual dataset
    full_dataset = datasets.ImageFolder(data_dir)
   
    # perform split
    train_size = int(0.8 * len(full_dataset))
    search_size = int(0.1 * len(full_dataset))
    test_size = len(full_dataset) - train_size - search_size
   
    train_data, search_data, test_data = random_split(
        full_dataset, [train_size, search_size, test_size]
    )

    # apply transforms
    train_data.dataset.transform = get_train_transforms
    search_data.dataset.transform = get_val_transforms
    test_data.dataset.transform = get_val_transforms

    # create loaders and return
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=4)
    search_loader = DataLoader(search_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return train_loader, search_loader, test_loader


In [6]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from torchvision import transforms
import pandas as pd
from PIL import Image

df = pd.read_csv('data/labels.csv')

# fiachra's transforms
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

# fiachra's dataset class
class RadiologyDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform
        
        # map string labels to numbers (0-121)
        self.labels = self.df['label'].astype('category').cat.codes.values
        self.class_names = self.df['label'].astype('category').cat.categories

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.df.iloc[idx]['filename'])
        image = Image.open(img_path).convert('RGB')
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# splitting into three sets - train, val, test
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=43
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label'],
    random_state=42
)

train_set = RadiologyDataset(train_df, image_dir="data/test_images", transform=train_transforms)
val_set = RadiologyDataset(val_df, image_dir="data/test_images", transform=val_transforms)
test_set = RadiologyDataset(test_df, image_dir="data/test_images", transform=val_transforms)

print('training set has', len(train_set), 'images')
print('validation set has', len(val_set), 'images')
print('final test set has', len(test_set), 'images')

training set has 7200 images
validation set has 900 images
final test set has 900 images


### Supernet training
Here we pull together the various functions to actually train the supernet model and save it.

In [7]:
''' search space:
input resolution = [128, 160, 190, 224]
depth = [2, 3, 4]
width = [0.65, 0.8, 1.0, 1.2]
expansion ratio = [3,4,6]
'''
import torch.optim as optim
from torch.optim.lr_scheduler import ConstantLR, CosineAnnealingLR, SequentialLR, LinearLR

# main function to orchestrate training
def main(config):
    # load pretrained model (baseline)
    pre_model = load_pre_model(config.pretrained_model_checkpoint, config.device, config.num_data_classes)
    
    # initialize our supernet model
    snet = Supernet(config.supernet_width, config.supernet_expansion, num_classes=config.num_data_classes)
    num_params = sum(p.numel() for p in snet.parameters())
    print("Parameters:", num_params)
    print("Estimated FP32 size:", num_params * 4 / 1024**3, "GB")  # estimates size of supernet
    
    snet.to(config.device)
    # print(Supernet)  # confirm object
    
    # optimizer and LR scheduler
    optimizer = optim.SGD(snet.parameters(), lr=0.01, momentum=0.9, weight_decay=4e-5)
    scheduler_warmup = ConstantLR(optimizer, factor=1.0, total_iters=config.warmup_epochs) # keep LR constant over warmup epochs
    scheduler_cosine = CosineAnnealingLR(optimizer, T_max=(config.epochs - config.warmup_epochs), eta_min=1e-4)  # decay 0.05 -> 0.0001
    scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_cosine],  milestones=[config.warmup_epochs])
    
    #train_loader, search_loader, test_loader = get_data_loaders(data_directory, batch_size = batch_size)
    train_loader = DataLoader(train_set, batch_size=config.batch_size, shuffle=True, num_workers=1)
    val_loader = DataLoader(val_set, batch_size=config.batch_size, shuffle=False, num_workers=1)
    best_loss = 1.0
    
    for epoch in range(config.epochs):        
        stats, phase = train_supernet(snet, pre_model, config.paths_sampled, train_loader, optimizer, epoch, config.warmup_epochs, config.temperature, config.alpha)
        
        # validate every 5 epochs
        if (epoch + 1) % 5 == 0:
            val_stats = validate_supernet(snet, val_loader, config)
            print(f"--- VALIDATION Epoch {epoch} ---")
            print(f"MAX Path: Acc {val_stats['MAX']['accuracy']:.2f}% | Loss {val_stats['MAX']['avg_loss']:.4f}")
            print(f"MIN Path: Acc {val_stats['MIN']['accuracy']:.2f}% | Loss {val_stats['MIN']['avg_loss']:.4f}")
        
        scheduler.step()
        
        # report epoch stats
        print(f"Epoch {epoch} | {phase} | LR: {scheduler.get_last_lr()[0]:.6f} | Loss: {stats['avg_loss']:.4f} | KL (Soft): {stats['avg_kl']:.4f} | Accuracy: {stats['accuracy']:.2f}%")
        
        # save every 5 epochs if loss is improving
        if (epoch+1)%5 == 0:
            save_supernet(snet, f'supernet_epoch_{epoch+1}.pth')

In [8]:
# define a ocnfig class for supernet training
class Config:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.pretrained_model_checkpoint = 'trained_models/resnet50_baseline/resnet50_baseline.pth'
        
        # supernet init params
        self.supernet_width = [0.65, 0.8, 1.0, 1.2]
        self.supernet_expansion = [1.5, 2.0, 3.0]
        self.supernet_depth = [2, 3, 4]
        self.num_data_classes = 122
        
        # training params
        self.data_directory = 'data/test_images'
        self.batch_size = 32
        self.epochs = 100
        self.warmup_epochs=30
        self.paths_sampled=20  # per greedyNAS
        self.peak_lr = 0.05
        
        # distillation loss params
        self.temperature = 2.0
        self.alpha = 0.5

In [9]:
from knowledge_distillation.model_init import Supernet
from knowledge_distillation.train_loop import DistillationLoss, sample_configs, train_supernet, validate_supernet
cfg = Config()
main(cfg)

Parameters: 191225329
Estimated FP32 size: 0.7123698629438877 GB
Epoch 0 | warm | LR: 0.010000 | Loss: 7.6819 | KL (Soft): 40.4642 | Accuracy: 17.06%
Epoch 1 | warm | LR: 0.010000 | Loss: 4.7471 | KL (Soft): 13.2594 | Accuracy: 19.68%
Epoch 2 | warm | LR: 0.010000 | Loss: 4.1215 | KL (Soft): 11.7321 | Accuracy: 24.21%
Epoch 3 | warm | LR: 0.010000 | Loss: 3.6823 | KL (Soft): 11.0206 | Accuracy: 28.49%
--- VALIDATION Epoch 4 ---
MAX Path: Acc 1.67% | Loss 64.6056
MIN Path: Acc 12.56% | Loss 7.3291
Epoch 4 | warm | LR: 0.010000 | Loss: 3.3694 | KL (Soft): 10.4441 | Accuracy: 32.58%
supernet saved to supernet_epoch_5.pth
Epoch 5 | warm | LR: 0.010000 | Loss: 3.1193 | KL (Soft): 9.9671 | Accuracy: 35.54%
Epoch 6 | warm | LR: 0.010000 | Loss: 2.8033 | KL (Soft): 9.5892 | Accuracy: 41.17%
Epoch 7 | warm | LR: 0.010000 | Loss: 2.6121 | KL (Soft): 9.0538 | Accuracy: 45.21%
Epoch 8 | warm | LR: 0.010000 | Loss: 2.4575 | KL (Soft): 8.8785 | Accuracy: 48.01%
--- VALIDATION Epoch 9 ---
MAX Path: A

KeyboardInterrupt: 

### Evaluating Training
We need to determine if our trained supernet is actually useful.  We will do this by comparing the largest and smallest paths through the network.  If the largest is better, we confirm that the weight slicing is working properly - the larger network has more information/context than the smaller one.

In [ ]:
import torch

def calibrate_bn(supernet, config, calibration_loader, device, batches=50):
    # update custom batchnorm for specific config
    supernet.train()
    with torch.no_grad():
        for i, (images, _) in enumerate(calibration_loader):
            if i >= batches:
                break
            images = images.to(device)
            # forward pass through the specific architecture
            _ = supernet(images, config)
    
    supernet.eval() # return to eval mode for actual validation

def verify_ranking(supernet, calibration_loader, search_loader, device):
    # compare max and min path - the largest and smallest through the network
    supernet.eval()
    
    max_config = {'res': 224, 'width': 1.2, 'depth': [4, 4, 4, 4], 'exp': [6, 6, 6, 6]}
    min_config = {'res': 128, 'width': 0.65, 'depth': [2, 2, 2, 2], 'exp': [3, 3, 3, 3]}
    configs = [("MAX", max_config), ("MIN", min_config)]
    results = {}

    print("start ranking check")
    
    for name, config in configs:
        # calibrate batch norm
        calibrate_bn(supernet, config, calibration_loader, device)
        
        # evaluate
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in search_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = supernet(images, config)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        acc = 100 * correct / total
        results[name] = acc
        print(f"Configuration {name}: Accuracy = {acc:.2f}%")

    # Final Verdict
    if results["MAX"] > results["MIN"]:
        print(f"\nSUCCESS: MAX ({results['MAX']:.2f}%) > MIN ({results['MIN']:.2f}%)")
        print(f"Margin: {results['MAX'] - results['MIN']:.2f}%")
    else:
        print("\nINVERTED: MIN accuracy better than MAX.")
        
    return results

# orchestrating our function
width = [0.65, 0.8, 1.0, 1.2]
snet = Supernet(width, num_classes=165)
checkpoint = torch.load('supernet_epoch_100.pth', map_location='cuda')
snet.load_state_dict(checkpoint['state_dict'])
snet.to('cuda')

search_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=4)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=4)

results = verify_ranking(snet, train_loader, search_loader, 'cuda')

In [ ]:
import pandas as pd

# convert training results to csv for visualization
data = {
    "epoch": [5,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100],
    "avg_loss": [
        6.328765869140625,
        7.079707145690918,
        6.257150173187256,
        5.150867462158203,
        7.0459747314453125,
        7.76951789855957,
        7.156027793884277,
        12.210516929626465,
        7.550355434417725,
        4.895843029022217,
        4.747811317443848,
        4.419175148010254,
        4.472318649291992,
        3.988067865371704,
        4.1329026222229,
        4.495228290557861,
        4.040039539337158,
        4.673028945922852,
        4.015764236450195,
        4.169543743133545
    ]
}

df = pd.DataFrame(data)
df.to_csv("loss_curve.csv", index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("loss_curve.csv")

plt.plot(df["epoch"], df["avg_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Average Loss")
plt.title("Training Loss vs Epoch")
plt.grid(True)
plt.axvline(x=15, color="red", linestyle="--", label="Greedy Search Start")

plt.show()

In [ ]:
import torch
print(torch.cuda.memory_summary())

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
# old LR scheduler
optimizer = optim.SGD(snet.parameters(), lr = 0.1, momentum=0.9, weight_decay=4e-5)
    scheduler_warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs)  # increases LR linearly, up to 0.1 as we warm up
    scheduler_cosine  = CosineAnnealingLR(optimizer, T_max=(epochs-warmup_epochs))  # decreases LR from 0.1 to 0 over remaining epochs
    scheduler = SequentialLR(
        optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[warmup_epochs]  # scheduler puts these together
    )
    


In [4]:
!git status

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   knowledge_distillation/main.ipynb
	modified:   knowledge_distillation/model_init.py
	modified:   knowledge_distillation/train_loop.py
	modified:   radimage_data_download.ipynb
	modified:   radimage_model_download.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	final_student_v1_epoch_10.pth
	final_student_v1_epoch_20.pth
	final_student_v1_epoch_30.pth
	final_student_v1_epoch_40.pth
	final_student_v1_epoch_50.pth
	final_student_v2_epoch_10.pth
	final_student_v2_epoch_20.pth
	final_student_v2_epoch_30.pth
	final_student_v2_epoch_40.pth
	final_student_v2_epoch_50.pth
	hardware_lut.json
	knowledge_distillation/supernet_v1_loss.csv
	loss_curve.csv
	supernet_epoch

In [5]:
!git add knowledge_distillation/main.ipynb knowledge_distillation/optimization.ipynb knowledge_distillation/model_init.py knowledge_distillation/optimization.ipynb knowledge_distillation/train_loop.py

In [6]:
!git commit -m "NAS+KD progress"

[main 0242ccb] NAS+KD progress
 3 files changed, 294 insertions(+), 357 deletions(-)


In [8]:
!git pull origin main

remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 38 (delta 17), reused 26 (delta 13), pack-reused 0 (from 0)
Unpacking objects: 100% (38/38), 690.68 KiB | 15.01 MiB/s, done.
From https://github.com/eoconn25/CS6423_knowledge_distillation_project
 * branch            main       -> FETCH_HEAD
   68cfa84..815a89b  main       -> origin/main
hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint: 
hint:   git config pull.rebase false  # merge
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint: 
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-only on the command line to override the configur

In [10]:
!git pull --rebase --autostash origin main

From https://github.com/eoconn25/CS6423_knowledge_distillation_project
 * branch            main       -> FETCH_HEAD
Created autostash: 5c96400
Applying autostash resulted in conflicts.
Your changes are safe in the stash.
You can run "git stash pop" or "git stash drop" at any time.
Successfully rebased and updated refs/heads/main.


In [13]:
!git diff --name-only origin/main..HEAD

knowledge_distillation/main.ipynb
knowledge_distillation/model_init.py
knowledge_distillation/optimization.ipynb
knowledge_distillation/train_loop.py


In [21]:
!git push origin main

Enumerating objects: 18, done.
Counting objects: 100% (18/18), done.
Delta compression using up to 96 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (13/13), 53.67 KiB | 17.89 MiB/s, done.
Total 13 (delta 7), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (7/7), completed with 2 local objects.
remote: error: GH013: Repository rule violations found for refs/heads/main.
remote: 
remote: - GITHUB PUSH PROTECTION
remote:   —————————————————————————————————————————
remote:     Resolve the following violations before pushing again
remote: 
remote:     - Push cannot contain secrets
remote: 
remote:     
remote:      (?) Learn how to resolve a blocked push
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push
remote:     
remote:     
remote:       —— GitHub Personal Access Token ——————————————————————
remote:       